[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/templates/51_label_smoothing_ce.ipynb)

# 🟢 Easy: Label Smoothing Cross-Entropy

**Label Smoothing Cross-Entropy** (Szegedy et al. 2016) を実装する。one-hot target を soften することで model が無限に自信を持つのを防ぐ。modern training recipe の定番。

> 💡 **どこで使う**：モダンな CE のデフォルト。one-hot を soft にして over-confident な予測を抑制、`ε=0.1` が定番。

<details>
<summary>📖 もっと詳しく</summary>

Szegedy 2016。ImageNet で +0.3–0.5% の効果、calibration error も改善（モデルの自信が実際の正答率に近づく）。

LLM の事前学習でも使う。`nn.CrossEntropyLoss(label_smoothing=0.1)` で一発、自前実装はその裏側を理解するための鍛錬。

</details>

### Smoothed target distribution
$$q'[c] = \begin{cases} 1 - \varepsilon + \varepsilon/K & c = y \\ \varepsilon/K & c \neq y \end{cases}$$

この target に対する標準 cross-entropy：
$$L = -\sum_c q'[c] \log p[c], \quad p = \text{softmax}(\text{logits})$$

### なぜ効く
- over-confident prediction を防ぐ（logit gap が bound される）
- 正則化として効く — ImageNet で ~0.3-0.5% 上がる定番
- calibration error が下がる


### Signature
```python
def label_smoothing_ce(logits, targets, smoothing=0.1):
    # logits: (B, K) raw scores
    # targets: (B,) class indices (long)
    # smoothing: ε in [0, 1)
    # Returns: scalar loss (mean over batch)
```

### Rules
- `nn.CrossEntropyLoss(label_smoothing=...)` や `F.cross_entropy(label_smoothing=...)` は **使わない**
- 数値安定性のため `F.log_softmax` を使う（softmax → log ではない）
- `smoothing=0` の時、標準 cross-entropy と完全一致すること
- batch の **mean** を return（scalar）
- Target dist: 正解クラスは `(1-ε) + ε/K`、それ以外は `ε/K`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def label_smoothing_ce(logits, targets, smoothing=0.1):
    pass  # log_softmax、smoothed target dist 構築、-sum(q * log_p) → mean

In [ ]:
import torch
torch.manual_seed(0)
logits = torch.randn(4, 10)
targets = torch.tensor([3, 1, 7, 0])
print('CE (ε=0):  ', label_smoothing_ce(logits, targets, smoothing=0.0).item())
print('CE (ε=0.1):', label_smoothing_ce(logits, targets, smoothing=0.1).item())

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("label_smoothing_ce")